# 22.2 Scikit-learn 管道:防数据泄漏的工程纪律 / Sklearn Pipelines: the discipline against data leakage

**中文**:上一节说到"要把预处理和模型打包在一起存"——**Pipeline** 就是实现这件事的工具,但它的意义远不止"打包方便"。它是**防止数据泄漏(data leakage)** 的工程纪律,而数据泄漏是数据科学中**最隐蔽、最致命、最常见的错误**:它让你的模型在评估时**看起来好得不真实**,一上线就现原形。本节用一个触目惊心的真实实验证明这一点:**在纯噪声数据(特征和标签毫无关系)上,一个常见的泄漏操作能凭空制造出 0.78 的"准确率"(真实应该是 0.5)**。然后我们展示 `Pipeline` + `ColumnTransformer` 如何从根本上堵住这个漏洞。这是从"会调 sklearn"到"工程上可信"的关键一课。
**English**: The last section said "package preprocessing and model together to save" — **Pipeline** is the tool for that, but its significance goes far beyond "convenient packaging." It is the engineering discipline that **prevents data leakage**, and data leakage is data science's **most insidious, most fatal, and most common error**: it makes your model **look unrealistically good** at evaluation, then fall apart the moment it goes live. This section proves it with a startling real experiment: **on pure-noise data (features utterly unrelated to labels), a common leakage operation manufactures 0.78 "accuracy" out of thin air (the truth should be 0.5)**. Then we show how `Pipeline` + `ColumnTransformer` fundamentally plug this hole. This is the key lesson from "can call sklearn" to "engineering-trustworthy."

---

**中文**:**数据泄漏(data leakage)是什么**:*在训练/评估时,模型"偷看"了它在真实预测时不该拥有的信息*——最常见的是**测试集的信息泄漏进了训练/预处理**。经典泄漏源:
**English**: **What is data leakage**: *during training/evaluation, the model "peeks at" information it wouldn't have at real prediction time* — most commonly **test-set information leaking into training/preprocessing**. Classic leakage sources:
- **中文**:**在划分训练/测试之前**做了"看全量数据"的预处理:标准化(用了全量的均值/方差)、特征选择(用了全量的标签)、缺失值填充(用了全量的统计量)、过采样(SMOTE 用了测试样本的邻居)。
  **Before splitting** train/test, doing preprocessing that "sees all data": scaling (using global mean/variance), feature selection (using all labels), imputation (using global statistics), oversampling (SMOTE using test samples' neighbors).
- **中文**:**核心原则**:任何"从数据里学到参数"的步骤(scaler 的均值、selector 选的特征、imputer 的中位数),**都必须只在训练折上 fit,再应用到验证/测试折**。手动做极易出错,而 **Pipeline 让这件事自动正确**:交叉验证时,Pipeline 的每个步骤都在每个训练折上重新 fit,绝不碰验证折。
  **Core principle**: any step that "learns parameters from data" (a scaler's mean, a selector's chosen features, an imputer's median) **must fit only on the training fold, then apply to validation/test folds**. Doing this by hand is error-prone, but a **Pipeline makes it automatically correct**: in cross-validation, each Pipeline step is refit on each training fold and never touches the validation fold.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 极高频, 面试送命题）**
> **中文**:**数据泄漏**=模型在训练/评估时用了真实预测时拿不到的信息→评估虚高、上线崩。**最常见来源**:①预处理在划分前做(scaler/imputer/特征选择/SMOTE 看了测试集)②用未来信息预测过去(时序穿越)③特征里混入了标签的代理(如"是否退款"预测"是否欺诈")④测试集参与调参/特征工程。**解药=Pipeline**:把所有"学参数"的步骤(scaler/encoder/selector/imputer/model)串进 Pipeline, 交叉验证时**每折只在训练部分 fit**→杜绝泄漏; 且一个对象端到端 fit/predict, 部署时预处理和模型完全一致(防训练-服务偏差, 接 22.1)。**ColumnTransformer**:对不同列用不同变换(数值→标准化, 类别→OneHot), 并入 Pipeline。**时序**:必须用 `TimeSeriesSplit`(不能随机划分)。面试金句:*"数据泄漏是评估虚高、上线崩的头号杀手, 根因是预处理/特征选择在划分前看了测试数据; 用 Pipeline 把预处理和模型串起来, 交叉验证时每折只在训练折 fit, 从机制上杜绝泄漏, 同时一个对象端到端保证训练-服务一致; 不同列用 ColumnTransformer, 时序用 TimeSeriesSplit。"*
> **English**: **Data leakage** = the model uses information at training/evaluation it wouldn't have at real prediction → inflated evaluation, live collapse. **Most common sources**: ① preprocessing done before the split (scaler/imputer/feature-selection/SMOTE saw the test set) ② using future to predict past (time-series lookahead) ③ a label proxy sneaking into features (e.g. "was refunded" to predict "is fraud") ④ test set involved in tuning/feature engineering. **Cure = Pipeline**: chain all "parameter-learning" steps (scaler/encoder/selector/imputer/model) into a Pipeline so cross-validation **fits each fold only on its training portion** → eliminates leakage; and one object does end-to-end fit/predict so preprocessing and model are identical at deployment (prevents training-serving skew, per 22.1). **ColumnTransformer**: apply different transforms to different columns (numeric → scale, categorical → OneHot), inside the Pipeline. **Time series**: must use `TimeSeriesSplit` (never random split). Interview line: *"Data leakage is the top killer of inflated evaluation and live collapse, rooted in preprocessing/feature-selection seeing test data before the split; use a Pipeline to chain preprocessing and model so cross-validation fits each fold only on its training portion, mechanically eliminating leakage, while one object end-to-end guarantees training-serving consistency; use ColumnTransformer for different columns and TimeSeriesSplit for time series."*


In [ ]:

# ============================================================
# 触目惊心的泄漏实验:纯噪声数据上, 泄漏能凭空造出高准确率 / leakage manufactures accuracy from pure noise
# 中文:X 是纯随机噪声, 和 y 没有任何真实关系。诚实评估的准确率应该 ≈ 0.5(瞎猜)。
#      但如果"在交叉验证之前"用全量数据(含测试标签)做特征选择, 就会选出一些"碰巧和 y 相关"的噪声特征→虚假高分。
# English: X is pure random noise, no real relation to y. Honest accuracy should be ≈ 0.5 (chance).
#      But if feature selection uses ALL data (incl. test labels) BEFORE cross-validation, it picks noise features that
#      spuriously correlate with y → fake high score.
# ============================================================
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
np.random.seed(1)
n, p = 200, 2000
X=np.random.randn(n, p); y=np.random.randint(0, 2, n)      # 纯噪声:特征与 y 无关 / pure noise: features unrelated to y

# ✗ 错误:先用全量数据选 20 个"最佳"特征(看了测试标签), 再交叉验证 / WRONG: select on ALL data first, then CV
X_leaked=SelectKBest(f_classif, k=20).fit_transform(X, y)  # 选择器看到了全部 y(包括将来的测试折)/ selector saw all y
leaked=cross_val_score(LogisticRegression(max_iter=500), X_leaked, y, cv=5).mean()

# ✓ 正确:把特征选择放进 Pipeline, 交叉验证时每折只在训练部分选特征 / RIGHT: selection inside the Pipeline
pipe=Pipeline([("select", SelectKBest(f_classif, k=20)),
               ("clf", LogisticRegression(max_iter=500))])
honest=cross_val_score(pipe, X, y, cv=5).mean()            # 每折的特征选择只见训练数据 / per-fold selection sees only train

print("真相:数据是纯噪声, 诚实准确率应 ≈ 0.5(瞎猜水平)")
print(f"✗ 泄漏(划分前选特征)  CV 准确率: {leaked:.3f}  ← 虚假的高分, 全是幻觉!")
print(f"✓ 正确(Pipeline 内选特征) CV 准确率: {honest:.3f}  ← 诚实地暴露'根本没有信号'")
print(f"\n泄漏凭空制造了 +{(leaked-honest)*100:.0f} 个百分点的假业绩——上线后必然打回原形")


In [ ]:

# ============================================================
# ColumnTransformer:对不同类型的列做不同预处理 / different preprocessing for different column types
# 中文:真实数据混合数值列和类别列。数值列要填充缺失+标准化, 类别列要独热编码。ColumnTransformer 分别处理,
#      再和模型串成一条 Pipeline——端到端 fit/predict, 部署时预处理与模型完全一致。
# English: real data mixes numeric and categorical columns. Numeric → impute + scale; categorical → one-hot.
#      ColumnTransformer handles each, then chains with the model into one Pipeline — end-to-end, consistent at deployment.
# ============================================================
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
df=pd.DataFrame({"age":[25,38,np.nan,55,29,41,np.nan,33],
                 "fare":[7.2,71.3,8.0,53.1,13.0,27.7,7.9,10.5],
                 "sex":["m","f","f","m","f","m","f","m"],
                 "embarked":["S","C","S","Q","S","C","S","S"],
                 "survived":[0,1,1,0,1,0,1,0]})
X, y = df.drop(columns="survived"), df["survived"]
num_cols, cat_cols = ["age","fare"], ["sex","embarked"]
# 数值管道:中位数填充 → 标准化 / numeric: median impute → scale
num_pipe=Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
# 组合:不同列走不同变换 / combine: different columns, different transforms
preprocess=ColumnTransformer([("num", num_pipe, num_cols),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])
# 完整端到端管道:预处理 + 模型 / full end-to-end pipeline
full=Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=500))]).fit(X, y)
print("端到端 Pipeline 训练完成 / end-to-end pipeline fitted. 一个对象搞定预处理+模型")
print("对原始 DataFrame 直接预测(自动填充→标准化→独热→预测)/ predict on raw DataFrame directly:")
print("  预测 / predictions:", full.predict(X))
import joblib; joblib.dump(full, "/tmp/full_pipeline.joblib")   # 存整条 pipeline(接 22.1)/ save the whole pipeline
print("  已保存整条 Pipeline → 部署时加载即用, 预处理与训练完全一致(防训练-服务偏差)")


In [ ]:

# ============================================================
# 可视化:泄漏 vs 诚实, 以及 Pipeline 在交叉验证中的正确行为 / leakage vs honest + pipeline in CV
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① 泄漏 vs 诚实 / leakage vs honest
bars=ax[0].bar(["✗ 泄漏\n(划分前选特征)","✓ 正确\n(Pipeline内选)","真实水平\n(纯噪声=瞎猜)"],
               [leaked,honest,0.5],color=["#C44E52","#55A868","#999999"])
for b,v in zip(bars,[leaked,honest,0.5]): ax[0].text(b.get_x()+b.get_width()/2,v+0.01,f"{v:.2f}",ha="center",fontsize=11,weight="bold")
ax[0].set_ylabel("交叉验证准确率"); ax[0].set_title("纯噪声数据:泄漏凭空造出假业绩")
ax[0].axhline(0.5,ls="--",color="gray",alpha=0.6)
# ② 正确的 CV:每折 fit 只在训练折 / correct CV: fit only on train fold
ax[1].axis("off"); ax[1].set_title("Pipeline 在交叉验证中的正确行为",fontsize=12,weight="bold")
for f in range(3):
    y0=0.72-f*0.22
    for k in range(5):
        c="#55A868" if k!=f else "#C44E52"; lab="训练(fit预处理+模型)" if k!=f else "验证(只transform)"
        ax[1].add_patch(plt.Rectangle((0.08+k*0.17,y0),0.15,0.13,fc=c,alpha=0.6,transform=ax[1].transAxes))
    ax[1].text(0.08,y0+0.15,f"折{f+1}",fontsize=8,transform=ax[1].transAxes)
ax[1].text(0.5,0.06,"每一折:预处理(scaler/selector)只在绿色训练块 fit, 红色验证块只 transform\n→ 验证块的信息绝不泄漏进预处理", ha="center",fontsize=8.5,style="italic",transform=ax[1].transAxes)
ax[1].scatter([],[],c="#55A868",label="训练折(fit)"); ax[1].scatter([],[],c="#C44E52",label="验证折(仅transform)")
ax[1].legend(loc="upper right",fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/mlops02_viz.png",dpi=80); plt.show()
print("左:泄漏造出 0.78 假准确率(真实=0.5); 右:Pipeline 让每折的预处理只见训练数据, 机制上杜绝泄漏")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **数据泄漏是数据科学的头号隐形杀手,而且极其常见**:我们在**纯噪声**数据上——特征和标签毫无关系,任何诚实的模型都只能瞎猜(0.5)——仅仅因为"在交叉验证之前用全量数据做了特征选择",就制造出了 **0.78 的假准确率**。想象一下:如果你在真实项目里犯了这个错,你会兴奋地向老板汇报"模型准确率 78%!",上线后却发现它和抛硬币一样——因为那 78% 完全是**特征选择器偷看了测试标签**造出的幻觉。这不是罕见的极端案例,而是**新手和老手都经常犯的错**(尤其在高维数据、特征选择、缺失值填充、SMOTE 过采样时)。
2. **Pipeline 不是"语法糖",而是把正确性写进机制**:很多人以为 Pipeline 只是"少写几行代码"的便利工具,其实它的核心价值是**从机制上保证不泄漏**:当 Pipeline 参与交叉验证时,它的每一个"学参数"的步骤(scaler、selector、imputer)都会在**每个训练折上重新 fit,绝不触碰验证折**。你手动做这件事——先 split、再对每一折分别 fit_transform——极其繁琐且容易漏;Pipeline 让它**自动、正确、无法搞错**。这就是为什么"任何预处理都应该在 Pipeline 里"是专业数据科学家的铁律。
3. **Pipeline 同时解决了 22.1 的训练-服务偏差**:一个额外的巨大好处——因为预处理和模型被打包成**一个对象**,你部署时只需 `joblib.load(pipeline).predict(raw_data)`,预处理逻辑和训练时**字节级一致**,根本不可能出现"线上手动复刻标准化时算错均值"这类训练-服务偏差。**诚实的边界**:①Pipeline 不能防所有泄漏——**时序穿越**(用未来预测过去)要靠 `TimeSeriesSplit` 而非随机划分;**标签代理特征**(如用"是否已退款"预测"是否欺诈")这类业务逻辑泄漏,Pipeline 也管不了,得靠对业务和数据生成过程的理解。②复杂 Pipeline 的调试和自定义 transformer 有学习成本。但这些都不改变结论:**把预处理放进 Pipeline,是数据科学工程可信度的底线**。**结论:泄漏让评估结果变成美丽的谎言,而 Pipeline 是把"诚实评估"写进代码机制的工程纪律——这也呼应了本仓库反复强调的诚实主线:一个数字如果好得不真实,先怀疑泄漏。**

**English**:
1. **Data leakage is data science's top invisible killer, and extremely common**: on **pure-noise** data — features utterly unrelated to labels, where any honest model can only guess (0.5) — merely "doing feature selection on all data before cross-validation" manufactured a **fake 0.78 accuracy**. Imagine: if you made this mistake in a real project, you'd excitedly report "78% accuracy!" to your boss, then find it performs like a coin flip in production — because that 78% was pure illusion from the **feature selector peeking at test labels**. This isn't a rare extreme case but **a mistake both novices and veterans frequently make** (especially with high-dimensional data, feature selection, imputation, SMOTE oversampling).
2. **Pipeline isn't "syntactic sugar" but writing correctness into the mechanism**: many think Pipeline is just a "write fewer lines" convenience, but its core value is **mechanically guaranteeing no leakage**: when a Pipeline participates in cross-validation, each of its "parameter-learning" steps (scaler, selector, imputer) is **refit on each training fold and never touches the validation fold**. Doing this by hand — split first, then fit_transform each fold separately — is tedious and error-prone; the Pipeline makes it **automatic, correct, and impossible to get wrong**. This is why "any preprocessing should be inside the Pipeline" is a professional data scientist's iron rule.
3. **Pipeline also solves 22.1's training-serving skew**: an extra huge benefit — because preprocessing and model are packaged into **one object**, deployment is just `joblib.load(pipeline).predict(raw_data)`, with preprocessing logic **byte-identical** to training, making "miscomputing the mean when manually replicating scaling online" impossible. **Honest limits**: ① Pipeline can't prevent all leakage — **time-series lookahead** (using future to predict past) needs `TimeSeriesSplit` not random split; **label-proxy features** (e.g. using "was refunded" to predict "is fraud") are business-logic leakage that Pipeline can't catch, requiring understanding of the business and data-generation process. ② Complex Pipelines and custom transformers have a learning cost. But none of this changes the conclusion: **putting preprocessing in a Pipeline is the baseline of engineering trustworthiness in data science**. **Conclusion: leakage turns evaluation into a beautiful lie, and Pipeline is the engineering discipline that writes "honest evaluation" into the code mechanism — echoing this repo's recurring honest thread: if a number is unrealistically good, suspect leakage first.**

> 💼 **实战视角 / Practical angle**
> **中文**:Pipeline 落地:①**所有预处理都进 Pipeline**(`ColumnTransformer` 分列处理 + 模型), 绝不在 `train_test_split` 之前 `fit_transform` 全量数据;②交叉验证/网格搜索直接传 Pipeline(`GridSearchCV(pipe, params)`), 让调参也无泄漏;③时序数据用 `TimeSeriesSplit`, 绝不随机划分;④**警惕业务逻辑泄漏**——检查每个特征"在真实预测那一刻是否已知"(未来信息、标签代理);⑤部署存整条 Pipeline(接 22.1), `predict` 直接吃原始数据;⑥自定义步骤继承 `BaseEstimator`+`TransformerMixin` 写成 transformer 放进 Pipeline。**红旗信号**:准确率高得离谱、某特征重要性异常高(可能是标签代理)、线上远差于离线(训练-服务偏差或泄漏)。面试金句:*"任何从数据学参数的预处理都必须只在训练折 fit; 用 Pipeline+ColumnTransformer 把预处理和模型串起来, 交叉验证每折独立 fit 从机制上防泄漏, 同时端到端一致防训练-服务偏差; 时序用 TimeSeriesSplit; 还要人工排查业务逻辑泄漏(未来信息/标签代理)。评估好得不真实?先查泄漏。"*
> **English**: Pipeline in practice: ① **all preprocessing into the Pipeline** (`ColumnTransformer` for per-column handling + model), never `fit_transform` on all data before `train_test_split`; ② pass the Pipeline directly to cross-validation/grid search (`GridSearchCV(pipe, params)`) so tuning is leakage-free too; ③ time-series data uses `TimeSeriesSplit`, never random split; ④ **beware business-logic leakage** — check each feature "is it known at the actual prediction moment" (future info, label proxies); ⑤ deploy by saving the whole Pipeline (per 22.1), `predict` on raw data directly; ⑥ custom steps inherit `BaseEstimator`+`TransformerMixin` as transformers in the Pipeline. **Red flags**: absurdly high accuracy, one feature's importance abnormally high (possible label proxy), online far worse than offline (training-serving skew or leakage). Interview line: *"Any preprocessing that learns parameters from data must fit only on the training fold; use Pipeline + ColumnTransformer to chain preprocessing and model so cross-validation fits each fold independently, mechanically preventing leakage while end-to-end consistency prevents training-serving skew; use TimeSeriesSplit for time series; also manually check for business-logic leakage (future info/label proxies). Evaluation unrealistically good? Check leakage first."*

---
### 小结 / Summary
- **中文**:数据泄漏(评估用了预测时拿不到的信息)让离线虚高、上线崩; 纯噪声上泄漏能造出 0.78 假准确率。
- **English**: Data leakage (evaluation uses info unavailable at prediction) inflates offline and collapses live; on pure noise it manufactures 0.78 fake accuracy.
- **中文**:Pipeline 把所有"学参数"步骤串起来, 交叉验证每折只在训练折 fit → 机制上杜绝泄漏; ColumnTransformer 分列处理。
- **English**: Pipeline chains all "parameter-learning" steps so cross-validation fits each fold only on train → mechanically prevents leakage; ColumnTransformer handles columns separately.
- **中文**:一个对象端到端 fit/predict 也防训练-服务偏差(接 22.1); 时序用 TimeSeriesSplit; 人工排查业务逻辑泄漏。
- **English**: One object end-to-end fit/predict also prevents training-serving skew (per 22.1); use TimeSeriesSplit for time series; manually check business-logic leakage.
